In [38]:
# multi linear reg
import numpy as np
import pandas as pd

df = pd.read_json("https://raw.githubusercontent.com/rajendra0968jangid/Ds-Arya/main/student-data.json")
print(df)

       name  gender  physics  maths  english
0      arun    Male       88     87       78
1    rajesh    Male       96    100       95
2   moorthy    Male       89     90       70
3      raja    Male       30     25       40
4      usha  Female       67     45       78
5     priya  Female       56     46       78
6    Sundar    Male       12     67       67
7   Kavitha  Female       78     71       67
8    Dinesh    Male       56     45       67
9      Hema  Female       71     90       23
10    Gowri  Female      100    100      100
11      Ram    Male       78     55       47
12  Murugan    Male       34     89       78
13  Jenifer  Female       67     88       90


In [54]:
import boto3
bucket = "aws-arya-s3-my-linear-regression-bucket"
key = "dataset/student-data.json"
s3 = boto3.client("s3")

obj = s3.get_object(Bucket=bucket,Key=key)
df = pd.read_json(obj["Body"])
print(df)

       name  gender  physics  maths  english
0      arun    Male       88     87       78
1    rajesh    Male       96    100       95
2   moorthy    Male       89     90       70
3      raja    Male       30     25       40
4      usha  Female       67     45       78
5     priya  Female       56     46       78
6    Sundar    Male       12     67       67
7   Kavitha  Female       78     71       67
8    Dinesh    Male       56     45       67
9      Hema  Female       71     90       23
10    Gowri  Female      100    100      100
11      Ram    Male       78     55       47
12  Murugan    Male       34     89       78
13  Jenifer  Female       67     88       90


In [40]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
# Independent Variables
X = df[['maths', 'english']]
 
# Dependent Variable
y = df['physics']
 
# Split Data
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)
 
# Create Model
model = LinearRegression()
 
# Train Model
model.fit(X_train, y_train)
 
# Prediction
y_pred = model.predict(X_test)
print(X_test)
print(y_pred)

    maths  english
9      90       23
11     55       47
0      87       78
[21.92515979 35.39054173 67.800019  ]


In [41]:
# model dump
import joblib
joblib.dump(model,"model.joblib")

['model.joblib']

In [42]:
s3.upload_file("model.joblib",bucket,"model/model.joblib")

In [43]:
# compress model
import tarfile
with tarfile.open('model.tar.gz' , 'w:gz') as tar:
    tar.add('model.joblib')

In [44]:
# compressed model push -> s3 bucket
s3.upload_file("model.tar.gz",bucket,"model/model.tar.gz")



In [59]:
!python inference.py


In [2]:
!pip install sagemaker==2.248.1

In [60]:
import sagemaker

print(sagemaker.__version__)

2.248.1


In [51]:
# deployment cell -> 10 min 
from sagemaker.sklearn.model import SKLearnModel
import sagemaker

session = sagemaker.Session()
role = sagemaker.get_execution_role()

sk_model = SKLearnModel(
    model_data= f"s3://{bucket}/model/model.tar.gz",
    role=role,
    framework_version="1.2-1",
    py_version="py3",
    entry_point="inference.py",
    # source_dir="code"
)

predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium"
)

print("Endpoint Name:", predictor.endpoint_name)

-

-

-

-

-

...

-

-

-

!

Endpoint Name: sagemaker-scikit-learn-2026-07-14-10-14-42-660


In [52]:
import boto3
import json

runtime = boto3.client("sagemaker-runtime")

payload = {
    "maths":40,
    "english":60
}

response = runtime.invoke_endpoint(
    EndpointName="sagemaker-scikit-learn-2026-07-14-10-14-42-660",
    ContentType="application/json",
    Body=json.dumps(payload)
)

print(response["Body"].read().decode())

{"prediction": [43.454169268701456]}
